# SKKB - Import traces (local / Databricks dual-mode)

Load MLflow traces for a single evaluation run and convert them into the same trace-level flat dataframe shape used by the SKKB data-preparation notebook.

This version runs **either**:
- on a Databricks cluster (`RUN_ON_DBX = True`) — uses `spark` / `dbutils` directly, **or**
- on a local machine (`RUN_ON_DBX = False`) — authenticates via the Databricks CLI profile / SDK, talks to the workspace MLflow tracking server and serving endpoints, and (optionally) uses Databricks Connect for Unity Catalog reads/writes.

All trace-derived evaluation fields come from the MLflow experiment traces. The helper English columns are enriched separately: `user_query_en` and persisted `expected_response_en` come from the SKKB test-set JSON, and `agent_response_en` comes from a persistent sidecar translation cache.

## Run mode

In [1]:
# Toggle to run on a Databricks cluster vs. locally from your laptop.
RUN_ON_DBX: bool = False

# Local-only: which Databricks CLI profile to use for auth (matches `databricks --profile <name>`).
DBX_PROFILE: str = "adb-chat"

# Local-only: whether to also write the parsed dataframe back to Unity Catalog
# (requires databricks-connect installed and matching the cluster DBR version).
WRITE_TO_UC: bool = True

In [2]:
!databricks auth describe --profile adb-chat   # OK
!databricks current-user me  --profile adb-chat # OK

]11;?\Host: https://adb-3174992876438447.7.azuredatabricks.net
User: sg7cb@s-mxs.net
Authenticated with: metadata-service
-----
Current configuration:
  ✓ host: https://adb-3174992876438447.7.azuredatabricks.net (from DATABRICKS_HOST environment variable)
  ✓ cluster_id: 0424-122149-m97focly (from DATABRICKS_CLUSTER_ID environment variable)
  ✓ metadata_service_url: ******** (from DATABRICKS_METADATA_SERVICE_URL environment variable)
  ✓ account_id: 85cde0e4-68ed-4a99-b884-abaec50d1f90 (from /Users/SG7CB/.databrickscfg config file)
  ✓ workspace_id: 3174992876438447 (from /Users/SG7CB/.databrickscfg config file)
  ✓ profile: adb-chat (from --profile flag)
  ✓ auth_type: metadata-service (from DATABRICKS_AUTH_TYPE environment variable)
  ✓ cloud: Azure
  ✓ audience: https://adb-3174992876438447.7.azuredatabricks.net/oidc/v1/token
  ✓ discovery_url: https://adb-3174992876438447.7.azuredatabricks.net/oidc/.well-known/oauth-authorization-server
]11;?\{
  "active":true,
  "displayName":

## Imports & auth bootstrap

When running locally we point MLflow at the Databricks tracking server using the CLI profile we already verified with `databricks auth describe --profile adb-chat`. No PAT needed.

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
import os as _os_grpc
_os_grpc.environ.setdefault("GRPC_VERBOSITY", "ERROR")
_os_grpc.environ.setdefault("GRPC_POLL_STRATEGY", "poll")
_os_grpc.environ.setdefault("GLOG_minloglevel", "2")

'2'

In [5]:
import os
import sys
import importlib
from pathlib import Path

# Make the local hg-ds-evals repo importable when running off-cluster.
# (config_nbs.add_local_paths() only adds /Workspace/... paths that exist on Databricks.)
if not globals().get("RUN_ON_DBX", False):
    _repo_root = Path.cwd()
    while _repo_root != _repo_root.parent and not (_repo_root / "hg_ds_evals").is_dir():
        _repo_root = _repo_root.parent
    if (_repo_root / "hg_ds_evals").is_dir() and str(_repo_root) not in sys.path:
        sys.path.insert(0, str(_repo_root))
        print(f'Local repo root added to sys.path: "{_repo_root}"')

import config_nbs
config_nbs.add_local_paths()

import mlflow
import pandas as pd
import hg_ds_evals.preprocessing.skkb_traces as skkb_traces

importlib.reload(skkb_traces)
build_skkb_dataframe_from_mlflow_search_traces = skkb_traces.build_skkb_dataframe_from_mlflow_search_traces

if not RUN_ON_DBX:
    # Make every Databricks SDK / MLflow call use the named CLI profile.
    os.environ["DATABRICKS_CONFIG_PROFILE"] = DBX_PROFILE

    from databricks.sdk import WorkspaceClient
    _w = WorkspaceClient()
    print(f"Authenticated as: {_w.current_user.me().user_name}")
    print(f"Workspace host:   {_w.config.host}")

    # Point MLflow at the Databricks-hosted tracking server.
    mlflow.set_tracking_uri("databricks")
    print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
else:
    print("Running on Databricks cluster — using attached spark/dbutils/MLflow.")

Path "/Workspace/Users/sg7cb@s-mxs.net/hg-ds-evals/" added!
Path "/Workspace/Users/sg7cb@s-mxs.net/hg-ds-evals/experiments/skkb/" added!
Authenticated as: sg7cb@s-mxs.net
Workspace host:   https://adb-3174992876438447.7.azuredatabricks.net
MLflow tracking URI: databricks


## Run / experiment configuration

In [6]:
all_runs:dict = {
    "adrian_test": {
        "experiment_id": "471458066277040",
        "run_id": "bf9b770ad4ce4324b58034d90b909867",
        "mlflow_run_name": "offline/adhoc/silver-wolf/infer",
    },
    "may_1": {
        "experiment_id": "471458066277040",
        "run_id": "db7ae47eee0348f8a3fd9c6dd21b5b3a",
        "mlflow_run_name": "online/nightly/2026-05-01/score",
    },

    "april_24": {
        "experiment_id": "471458066277040",
        "run_id": "3441808bc8db416fbbce3f4878e94d4d",
        "mlflow_run_name": "online_nightly_2026_04_24_infer",
    },
}

In [7]:
RUN_DATE = "adrian_test"
EXPERIMENT_ID = all_runs[RUN_DATE]["experiment_id"]
RUN_ID = all_runs[RUN_DATE]["run_id"]
RUN_NAME = all_runs[RUN_DATE]["mlflow_run_name"].replace("/", "_").replace("-", "_")
print(f"Selected run:"
      f"\nExperiment id: {EXPERIMENT_ID}"
      f"\nRun name: {RUN_NAME}"
      f"\nRun id: {RUN_ID}")

MAX_RESULTS = None

TEST_SET_JSON_PATH = "../input/test_set_513_2026-04-16_14h54_GAI_SK_SK.json"
KB_SK_CSV_PATH = "../input/KB_GAI_SK_SK_2026-04-20_14h16_phase_1_2.csv"
KB_EN_CSV_PATH = "../input/KB_GAI_SK_EN_2026-04-20_14h16_phase_1_2.csv"

DBX_CATALOG: str = "prod_aut_chat00_lab_catalog"
DBX_SCHEMA: str = "private_uc0115_aix_db"
DBX_TABLE: str = f"dts_eval_skkb_exp_001_{RUN_NAME}"
print(f"Databricks Unity Catalog target: {DBX_CATALOG}.{DBX_SCHEMA}.{DBX_TABLE}")

DBX_CLUSTER_ID: str = "0424-122149-m97focly"
# Local-only: cluster id used by Databricks Connect (must be a running interactive cluster).

Selected run:
Experiment id: 471458066277040
Run name: offline_adhoc_silver_wolf_infer
Run id: bf9b770ad4ce4324b58034d90b909867
Databricks Unity Catalog target: prod_aut_chat00_lab_catalog.private_uc0115_aix_db.dts_eval_skkb_exp_001_offline_adhoc_silver_wolf_infer


## Fetch traces from MLflow

Works identically locally and on a Databricks cluster — the only difference is the tracking URI we set above.

In [8]:
import logging

# Quiet the noisy "Connection pool is full" warnings emitted while MLflow
# downloads trace artifacts from blob storage in parallel.
logging.getLogger("urllib3.connectionpool").setLevel(logging.ERROR)

traces_df = mlflow.search_traces(
    locations=[EXPERIMENT_ID],   # `experiment_ids` is deprecated in MLflow 3.x
    run_id=RUN_ID,
    max_results=MAX_RESULTS,
    order_by=["timestamp_ms DESC"],
)

print(f"Shape: {traces_df.shape}")
print(f"Columns: {traces_df.columns.tolist()}")
traces_df.head()

KeyboardInterrupt: 

In [9]:
parse_result = build_skkb_dataframe_from_mlflow_search_traces(traces_df)

trace_level_df = parse_result.dataframe
parse_errors = parse_result.parse_errors
unmapped_trace_ids = parse_result.unmapped_trace_ids

print(f"Run: {RUN_NAME}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Trace rows fetched: {len(traces_df):,}")
print(f"Parsed rows: {len(trace_level_df):,}")
print(f"Parse errors: {len(parse_errors):,}")
print(f"Rows using trace_id as test_case_id fallback: {len(unmapped_trace_ids):,}")

if parse_errors:
    print(parse_errors[:5])

Run: online_nightly_2026_05_01_score
Tracking URI: databricks
Trace rows fetched: 513
Parsed rows: 513
Parse errors: 0
Rows using trace_id as test_case_id fallback: 1


## Add enrichment columns and persistent English text caches

Get the host + token for the translation HTTP calls — locally we use the Databricks SDK instead of `spark.conf` / `dbutils`.

In [10]:
import concurrent.futures
import json
from pathlib import Path
from urllib import error, request

TRANSLATE_MODEL = "gpt-5-1"
TRANSLATE_MAX_CONCURRENCY = 20
TRANSLATE_MAX_OUTPUT_TOKENS = 1024

TRANSLATE_SYSTEM_PROMPT = (
    "You are a professional Slovak-to-English translator. Translate the user "
    "message correctly and concisely. Preserve banking terminology, product "
    "names, numbers, and formatting. Do NOT add commentary, quotes, or "
    "explanations. Output ONLY the English translation."
)

# Per-run translation cache: SK->EN for user_query / expected_response /
# agent_response. Lives next to the input data, named after the run so it
# stays connected to that specific MLflow run. On re-import of the same run
# it is a pure cache hit; for a new run it is built lazily.
TRANSLATIONS_CACHE_PATH = f"../input/skkb_translations_{RUN_NAME}.json"

# Resolve workspace host + bearer token in a way that works both on the cluster and locally.
if RUN_ON_DBX:
    _dbx_host = spark.conf.get("spark.databricks.workspaceUrl")  # noqa: F821 (provided by DBR)
    _dbx_token = (
        dbutils.notebook.entry_point.getDbutils()  # noqa: F821 (provided by DBR)
        .notebook()
        .getContext()
        .apiToken()
        .get()
    )
else:
    from databricks.sdk import WorkspaceClient
    _w_local = WorkspaceClient()
    _dbx_host = _w_local.config.host.replace("https://", "").rstrip("/")
    _dbx_token = _w_local.config.authenticate()["Authorization"].removeprefix("Bearer ")

_translate_url = f"https://{_dbx_host}/serving-endpoints/chat/completions"


def _atomic_write_json(path: str, payload: dict) -> None:
    target_path = Path(path)
    target_path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = target_path.with_suffix(target_path.suffix + ".tmp")
    with temp_path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, ensure_ascii=False, indent=2, sort_keys=True)
    temp_path.replace(target_path)


def _load_json(path: str, default: dict) -> dict:
    try:
        with open(path, "r", encoding="utf-8") as handle:
            payload = json.load(handle)
    except FileNotFoundError:
        return default
    return payload if isinstance(payload, dict) else default


def _translate_one(text: str) -> str:
    payload = {
        "model": TRANSLATE_MODEL,
        "messages": [
            {"role": "system", "content": TRANSLATE_SYSTEM_PROMPT},
            {"role": "user", "content": text},
        ],
        "max_tokens": TRANSLATE_MAX_OUTPUT_TOKENS,
    }
    req = request.Request(
        _translate_url,
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Authorization": f"Bearer {_dbx_token}",
            "Content-Type": "application/json",
        },
        method="POST",
    )
    try:
        with request.urlopen(req, timeout=120) as response:
            response_payload = json.loads(response.read().decode("utf-8"))
    except error.HTTPError as exc:
        detail = exc.read().decode("utf-8", errors="ignore")
        raise RuntimeError(f"Translation request failed for text {text[:80]!r}: {detail}") from exc

    choice = ((response_payload.get("choices") or [{}])[0].get("message") or {})
    return (choice.get("content") or "").strip()


def _translate_unique_texts(texts: list[str]) -> dict[str, str]:
    unique_texts = sorted({text for text in texts if isinstance(text, str) and text.strip()})
    if not unique_texts:
        return {}
    max_workers = min(TRANSLATE_MAX_CONCURRENCY, len(unique_texts))
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        translations = list(executor.map(_translate_one, unique_texts))
    return dict(zip(unique_texts, translations))


def _attach_kb(ids, lang_table: dict[str, dict[str, str]]) -> list[dict[str, object]]:
    attached = []
    for enum_id in ids if isinstance(ids, list) else []:
        enum_key = str(enum_id)
        kb_row = lang_table.get(enum_key)
        attached.append({
            "enum_id": enum_key,
            "description": (kb_row or {}).get("kb.description", ""),
            "missing": kb_row is None,
        })
    return attached


# -- Test-set JSON: ONLY for test_case_id resolution + categories_list -------
# (Translations live in our own per-run cache below -- the test-set EN fields
#  are intentionally not used and never overwritten.)
for required_path in (TEST_SET_JSON_PATH, KB_SK_CSV_PATH, KB_EN_CSV_PATH):
    if not Path(required_path).exists():
        raise FileNotFoundError(f"required enrichment file not found: {required_path}")

test_set_payload = _load_json(TEST_SET_JSON_PATH, {})
user_query_to_test_case_id = {}
for test_case_id, record in test_set_payload.items():
    if not isinstance(record, dict):
        continue
    user_query = record.get("user_query")
    if isinstance(user_query, str) and user_query.strip() and user_query not in user_query_to_test_case_id:
        user_query_to_test_case_id[user_query] = test_case_id


def _resolve_test_case_id(row) -> str | None:
    test_case_id = row.get("test_case_id")
    if isinstance(test_case_id, str) and test_case_id in test_set_payload:
        return test_case_id
    user_query = row.get("user_query")
    if isinstance(user_query, str):
        return user_query_to_test_case_id.get(user_query)
    return None


resolved_test_case_ids = trace_level_df.apply(_resolve_test_case_id, axis=1)
matched_test_set_rows = int(resolved_test_case_ids.notna().sum())
print(f"test-set matches: {matched_test_set_rows}/{len(trace_level_df)}")

trace_level_df["categories_list"] = resolved_test_case_ids.apply(
    lambda test_case_id: ((test_set_payload.get(test_case_id) or {}).get("categories_list") or []) if isinstance(test_case_id, str) else []
)

# -- KB descriptions (SK + EN) -- same as before -----------------------------
reranked_enum_values = trace_level_df["reranked_enum_ids"].tolist() if "reranked_enum_ids" in trace_level_df.columns else [[] for _ in range(len(trace_level_df))]
post_prune_enum_values = trace_level_df["post_prune_enum_ids"].tolist() if "post_prune_enum_ids" in trace_level_df.columns else [[] for _ in range(len(trace_level_df))]

kb_sk = pd.read_csv(KB_SK_CSV_PATH, sep="|", dtype=str).fillna("")
kb_en = pd.read_csv(KB_EN_CSV_PATH, sep="|", dtype=str).fillna("")
sk_by_id = kb_sk.set_index("kb.knowledgeId").to_dict(orient="index")
en_by_id = kb_en.set_index("kb.knowledgeId").to_dict(orient="index")
trace_level_df["reranked_enums_kb_sk"] = [_attach_kb(ids, sk_by_id) for ids in reranked_enum_values]
trace_level_df["reranked_enums_kb_en"] = [_attach_kb(ids, en_by_id) for ids in reranked_enum_values]
trace_level_df["post_prune_candidates_kb_sk"] = [_attach_kb(ids, sk_by_id) for ids in post_prune_enum_values]
trace_level_df["post_prune_candidates_kb_en"] = [_attach_kb(ids, en_by_id) for ids in post_prune_enum_values]

# -- Translations: load per-run cache, translate misses, save if dirty -------
print(f"\ntranslation cache: {TRANSLATIONS_CACHE_PATH}")
translations_cache = _load_json(
    TRANSLATIONS_CACHE_PATH,
    {"user_query": {}, "expected_response": {}, "agent_response": {}},
)
for section in ("user_query", "expected_response", "agent_response"):
    translations_cache.setdefault(section, {})


def _translate_field(field_name: str, source_values: list[str], cache: dict) -> tuple[list[str], bool]:
    section = cache[field_name]
    unique_inputs = sorted({t for t in source_values if isinstance(t, str) and t.strip()})
    missing = [t for t in unique_inputs if t not in section]
    print(f"  {field_name:>18s}_en: cached {len(unique_inputs) - len(missing)}/{len(unique_inputs)}"
          f", to translate {len(missing)}")
    if missing:
        section.update(_translate_unique_texts(missing))
    out = [section.get(t, "") if isinstance(t, str) and t.strip() else "" for t in source_values]
    return out, bool(missing)


user_query_values = trace_level_df["user_query"].tolist() if "user_query" in trace_level_df.columns else [""] * len(trace_level_df)
expected_response_values = trace_level_df["expected_response"].tolist() if "expected_response" in trace_level_df.columns else [""] * len(trace_level_df)
agent_response_values = trace_level_df["agent_response"].tolist() if "agent_response" in trace_level_df.columns else [""] * len(trace_level_df)

dirty = False
for field, values, target_col in (
    ("user_query",         user_query_values,         "user_query_en"),
    ("expected_response",  expected_response_values,  "expected_response_en"),
    ("agent_response",     agent_response_values,     "agent_response_en"),
):
    translated, did_translate = _translate_field(field, values, translations_cache)
    trace_level_df[target_col] = translated
    dirty = dirty or did_translate

if dirty:
    _atomic_write_json(TRANSLATIONS_CACHE_PATH, translations_cache)
    print(f"saved translations cache -> {TRANSLATIONS_CACHE_PATH}")
else:
    print("translations cache unchanged (all hits cached).")

print(f"\nuser_query_en non-empty:        {(trace_level_df['user_query_en'].str.len() > 0).sum()}/{len(trace_level_df)}")
print(f"expected_response_en non-empty: {(trace_level_df['expected_response_en'].str.len() > 0).sum()}/{len(trace_level_df)}")
print(f"agent_response_en non-empty:    {(trace_level_df['agent_response_en'].str.len() > 0).sum()}/{len(trace_level_df)}")
print(f"categories_list populated:      {sum(bool(v) for v in trace_level_df['categories_list'])}/{len(trace_level_df)}")
print(f"reranked_enums_kb_sk populated: {sum(bool(v) for v in trace_level_df['reranked_enums_kb_sk'])}/{len(trace_level_df)}")
print(f"post_prune_candidates_kb_sk populated: {sum(bool(v) for v in trace_level_df['post_prune_candidates_kb_sk'])}/{len(trace_level_df)}")


test-set matches: 513/513

translation cache: ../input/skkb_translations_online_nightly_2026_05_01_score.json
          user_query_en: cached 0/483, to translate 483
   expected_response_en: cached 0/512, to translate 512
      agent_response_en: cached 0/512, to translate 512
saved translations cache -> ../input/skkb_translations_online_nightly_2026_05_01_score.json

user_query_en non-empty:        484/513
expected_response_en non-empty: 512/513
agent_response_en non-empty:    512/513
categories_list populated:      513/513
reranked_enums_kb_sk populated: 424/513
post_prune_candidates_kb_sk populated: 424/513


In [11]:
trace_level_df.iloc[0][["user_query", "user_query_en", "agent_response", "agent_response_en", "expected_response", "expected_response_en"]]

user_query              Potrebujem niečo riešiť ohľadom účtu, môžem na...
user_query_en           I need to sort something out regarding my acco...
agent_response          Áno, svojmu bankovému poradcovi môžete napísať...
agent_response_en       Yes, you can also write directly to your bank ...
expected_response       Ospravedlňujem sa, nemám k dispozícii informác...
expected_response_en    I apologize, I do not have information availab...
Name: 0, dtype: object

In [12]:
if not RUN_ON_DBX:
    trace_level_df.to_csv(f"traces/skkb_traces_enriched_{RUN_NAME}.csv", index=False)

## View data

In [13]:
# `display(...)` is a Databricks-only helper; fall back to a plain dataframe locally.
if RUN_ON_DBX:
    display(trace_level_df)  # noqa: F821 (provided by DBR)
else:
    trace_level_df

In [14]:
trace_level_df.columns

Index(['test_case_id', 'trace_id', 'request_time', 'execution_duration_ms',
       'user_query', 'knowledge_search_run_count',
       'knowledge_search_final_run_index', 'knowledge_search_runs',
       'reranked_kb_context', 'kb_version', 'reranked_enum_ids',
       'raw_vector_db_retrieved_candidates_text',
       'raw_vector_db_retrieved_enum_ids',
       'raw_vector_db_retrieved_enum_count', 'raw_vector_db_query_count',
       'raw_vector_db_retrieved_count_by_query', 'raw_vector_db_query_limits',
       'pre_prune_candidates_text', 'pre_prune_enum_ids',
       'pre_prune_enum_count', 'post_prune_candidates_text',
       'post_prune_enum_ids', 'post_prune_enum_count',
       'prune_counts_available', 'prune_candidates_in', 'prune_candidates_out',
       'prune_candidates_dropped', 'agent_response', 'expected_response',
       'expected_enums', 'expected_enums_weights', 'expert_score',
       'expert_rationale', 'enum_relevance_score', 'agents_called',
       'agents_only', 'last_age

In [15]:
['test_case_id', 'trace_id', 'request_time', 'execution_duration_ms',
       'user_query', 'knowledge_search_run_count',
       'knowledge_search_final_run_index', 'knowledge_search_runs',
       'reranked_kb_context', 'kb_version', 'reranked_enum_ids',
       'raw_vector_db_retrieved_candidates_text',
       'raw_vector_db_retrieved_enum_ids',
       'raw_vector_db_retrieved_enum_count', 'raw_vector_db_query_count',
       'raw_vector_db_retrieved_count_by_query', 'raw_vector_db_query_limits',
       'pre_prune_candidates_text', 'pre_prune_enum_ids',
       'pre_prune_enum_count', 'post_prune_candidates_text',
       'post_prune_enum_ids', 'post_prune_enum_count',
       'prune_counts_available', 'prune_candidates_in', 'prune_candidates_out',
       'prune_candidates_dropped', 'agent_response', 'expected_response',
       'expected_enums', 'expected_enums_weights', 'expert_score',
       'expert_rationale', 'enum_relevance_score', 'agents_called',
       'agents_only', 'last_agent', 'tools_called', 'query_scope',
       'trace_invariant_violations', 'reranker_selected_empty',
       'reranker_raw_selected_ids', 'reranker_valid_selected_ids',
       'reranker_invalid_selected_ids', 'reranker_unselected_context_ids',
       'reranker_selection_status', 'reranker_selection_violations',
       'reranker_system_prompt', 'reranker_user_prompt', 'reranker_model',
       'reranker_token_usage', 'main_agent_prompt_hash',
       'daily_banking_agent_prompt_hash', 'user_query_en', 'categories_list',
       'expected_response_en', 'reranked_enums_kb_sk', 'reranked_enums_kb_en',
       'post_prune_candidates_kb_sk', 'post_prune_candidates_kb_en',
       'agent_response_en', 'execution_duration_s', 'execution_duration_min']

['test_case_id',
 'trace_id',
 'request_time',
 'execution_duration_ms',
 'user_query',
 'knowledge_search_run_count',
 'knowledge_search_final_run_index',
 'knowledge_search_runs',
 'reranked_kb_context',
 'kb_version',
 'reranked_enum_ids',
 'raw_vector_db_retrieved_candidates_text',
 'raw_vector_db_retrieved_enum_ids',
 'raw_vector_db_retrieved_enum_count',
 'raw_vector_db_query_count',
 'raw_vector_db_retrieved_count_by_query',
 'raw_vector_db_query_limits',
 'pre_prune_candidates_text',
 'pre_prune_enum_ids',
 'pre_prune_enum_count',
 'post_prune_candidates_text',
 'post_prune_enum_ids',
 'post_prune_enum_count',
 'prune_counts_available',
 'prune_candidates_in',
 'prune_candidates_out',
 'prune_candidates_dropped',
 'agent_response',
 'expected_response',
 'expected_enums',
 'expected_enums_weights',
 'expert_score',
 'expert_rationale',
 'enum_relevance_score',
 'agents_called',
 'agents_only',
 'last_agent',
 'tools_called',
 'query_scope',
 'trace_invariant_violations',
 'rera

In [16]:
trace_level_df["last_agent"].value_counts(dropna=False)

last_agent
daily_banking_agent    430
main_agent              83
Name: count, dtype: int64

In [17]:
cols = ["test_case_id",
        "user_query", "user_query_en",
        "raw_vector_db_retrieved_enum_count", "pre_prune_enum_count", "post_prune_enum_count", "reranked_enum_ids", "expected_enums",
        "reranker_raw_selected_ids", "reranker_valid_selected_ids", "reranker_invalid_selected_ids", "reranker_unselected_context_ids", "reranker_selection_status", "reranker_selection_violations"]
missing_cols = [column for column in cols if column not in trace_level_df.columns]
if missing_cols:
    print(f"Skipping missing preview columns: {missing_cols}")
cols = [column for column in cols if column in trace_level_df.columns]
trace_level_df[cols].sort_values(by="test_case_id", ascending=True).head()

,test_case_id,user_query,user_query_en,raw_vector_db_retrieved_enum_count,pre_prune_enum_count,post_prune_enum_count,reranked_enum_ids,expected_enums,reranker_raw_selected_ids,reranker_valid_selected_ids,reranker_invalid_selected_ids,reranker_unselected_context_ids,reranker_selection_status,reranker_selection_violations
512,Test case 0,Chcem si poslať peniaze zo sporenia späť na úč...,I want to transfer money from my savings back ...,180,77,35,"[SAVING@ACCOUNT_MOVEMENTS, START_TRANSFER]",[SAVING@ACCOUNT_MOVEMENTS],"[""SAVING@ACCOUNT_MOVEMENTS"", ""START_TRANSFER""]","[""SAVING@ACCOUNT_MOVEMENTS"", ""START_TRANSFER""]",[],[],ok,[]
511,Test case 1,"Dá sa na sporenie vložiť aj hotovosť, alebo le...","Can I deposit cash into the savings account, o...",180,59,31,"[SAVING@ACCOUNT_MOVEMENTS, SAVING@ABOUT]",[SAVING@ACCOUNT_MOVEMENTS],"[""SAVING@ACCOUNT_MOVEMENTS"", ""SAVING@ABOUT""]","[""SAVING@ACCOUNT_MOVEMENTS"", ""SAVING@ABOUT""]",[],[],ok,[]
502,Test case 10,Koľko sporení si môžem naraz vytvoriť a v akej...,How many savings accounts can I create at once...,180,74,35,[SAVING@ABOUT],[SAVING@ABOUT],"[""SAVING@ABOUT""]","[""SAVING@ABOUT""]",[],[],ok,[]
412,Test case 100,"Kde si viem pozrieť, či sa mi odmena vôbec zae...",Where can I see whether the reward has even be...,180,85,35,[DATEIO@ABOUT],[DATEIO@CLAIMS],"[""DATEIO@ABOUT""]","[""DATEIO@ABOUT""]",[],[],ok,[]
411,Test case 101,Ako reklamujem Moneyback?,How do I file a complaint about Moneyback?,180,57,35,"[DATEIO@CLAIMS, DATEIO@ABOUT]",[DATEIO@CLAIMS],"[""DATEIO@CLAIMS"", ""DATEIO@ABOUT""]","[""DATEIO@CLAIMS"", ""DATEIO@ABOUT""]",[],[],ok,[]


In [18]:
for enum_column in ("post_prune_enum_ids", "pre_prune_enum_ids"):
    if enum_column in trace_level_df.columns:
        print(f"empty {enum_column}:", sum(trace_level_df[enum_column].apply(lambda x: x == [])))
    else:
        print(f"{enum_column} missing")
count_cols = [
    "knowledge_search_run_count",
    "raw_vector_db_query_count",
    "raw_vector_db_retrieved_enum_count",
    "pre_prune_enum_count",
    "post_prune_enum_count",
    "prune_candidates_in",
    "prune_candidates_out",
    "prune_candidates_dropped",
]
count_cols = [column for column in count_cols if column in trace_level_df.columns]
if count_cols:
    trace_level_df[["test_case_id", *count_cols]].describe()

empty post_prune_enum_ids: 89
empty pre_prune_enum_ids: 89


### Execution time stats

In [19]:
trace_level_df["execution_duration_s"] = trace_level_df["execution_duration_ms"] / 1000
trace_level_df["execution_duration_min"] = trace_level_df["execution_duration_ms"] / (1000 * 60)

trace_level_df["execution_duration_s"].describe()

count    513.000000
mean      29.850283
std       13.564304
min        1.948000
25%       20.610000
50%       30.742000
75%       38.165000
max       74.674000
Name: execution_duration_s, dtype: float64

## Save parsed traces to Unity Catalog (optional)

Local execution requires `databricks-connect==<your-DBR-version>`. Set `WRITE_TO_UC = True` (and `RUN_ON_DBX = False` for the local path) to enable.

In [20]:
# Write the table to UC
if not WRITE_TO_UC:
    print("WRITE_TO_UC=False — skipping Unity Catalog write.")
elif trace_level_df.empty:
    raise ValueError("trace_level_df is empty; nothing to write")
else:
    trace_level_to_write = trace_level_df.copy()

    json_string_columns = [
        "reranked_enum_ids",
        "expected_enums",
        "knowledge_search_runs",
        "raw_vector_db_retrieved_enum_ids",
        "raw_vector_db_retrieved_count_by_query",
        "raw_vector_db_query_limits",
        "pre_prune_enum_ids",
        "post_prune_enum_ids",
        "categories_list",
        "reranked_enums_kb_sk",
        "reranked_enums_kb_en",
        "post_prune_candidates_kb_sk",
        "post_prune_candidates_kb_en",
        "agents_called",
        "tools_called",
        "trace_invariant_violations",
    ]

    for column in json_string_columns:
        if column in trace_level_to_write.columns:
            trace_level_to_write[column] = trace_level_to_write[column].apply(
                lambda value: json.dumps(value, ensure_ascii=False)
                if isinstance(value, (list, dict))
                else (value if isinstance(value, str) else "[]")
            )

    # Force a stable nullable-float dtype for assessment-derived numerics. Without
    # this, a run that skipped a scorer (e.g. all-None expert_score) lands as
    # object dtype → Spark NullType → column dropped under overwriteSchema=true
    # → any UC row filter / column mask referencing it then fails on read.
    nullable_float_cols = ("expert_score", "enum_relevance_score")
    for column in nullable_float_cols:
        if column in trace_level_to_write.columns:
            trace_level_to_write[column] = (
                pd.to_numeric(trace_level_to_write[column], errors="coerce").astype("Float64")
            )

    # Fail fast if a column the schema/policy depends on ever goes missing.
    required_cols = {"test_case_id", "trace_id", "expert_score", "expert_rationale", "enum_relevance_score"}
    missing = required_cols - set(trace_level_to_write.columns)
    if missing:
        raise ValueError(f"refusing to write — missing required columns: {sorted(missing)}")

    if RUN_ON_DBX:
        spark_session = spark  # noqa: F821 (provided by DBR)
    else:
        # Requires `databricks-connect` matching the cluster's DBR version.
        from databricks.connect import DatabricksSession
        spark_session = (
            DatabricksSession.builder
            .profile(DBX_PROFILE)
            .clusterId(DBX_CLUSTER_ID)
            .getOrCreate()
        )

    sdf = spark_session.createDataFrame(trace_level_to_write)
    row_count = len(trace_level_to_write)

    full_table = f"{DBX_CATALOG}.{DBX_SCHEMA}.{DBX_TABLE}"
    (
        sdf
        .write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_table)
    )

    print(f"wrote {row_count} rows to {full_table}")

wrote 513 rows to prod_aut_chat00_lab_catalog.private_uc0115_aix_db.dts_eval_skkb_exp_001_online_nightly_2026_05_01_score


In [21]:
# Check that the table was written correctly by reading it back and comparing row counts.
if WRITE_TO_UC:
    if RUN_ON_DBX:
        spark_session = spark  # noqa: F821 (provided by DBR)
    else:
        from databricks.connect import DatabricksSession
        spark_session = (
            DatabricksSession.builder
            .profile(DBX_PROFILE)
            .clusterId(DBX_CLUSTER_ID)
            .getOrCreate()
        )

    full_table = f"{DBX_CATALOG}.{DBX_SCHEMA}.{DBX_TABLE}"
    read_back_df = spark_session.table(full_table)
    read_back_count = read_back_df.count()
    original_count = len(trace_level_df)
    if read_back_count != original_count:
        raise ValueError(f"Row count mismatch: wrote {original_count} rows but read back {read_back_count} rows from {full_table}")
    else:
        print(f"Row count verified: {read_back_count} rows in {full_table}")

Row count verified: 513 rows in prod_aut_chat00_lab_catalog.private_uc0115_aix_db.dts_eval_skkb_exp_001_online_nightly_2026_05_01_score
